In [1]:
# Stage 5 — PPE Representation Study
# 5.2 — Probability Flowering Matrix (B)
#
# Goal: Replace binary plant existence matrix F with a continuous probability
# matrix PMf, where PMf[species, bin] = mean norm across 52 weeks from PPE.
# PCA PMf to 15D → Vf_prob. Pair with existing binary Vp.
# Feature vector: [Vf_prob (15D), Vp (15D), N (1D)] = 31D
# Compare against A2 (binary Vf + binary Vp) and A3 (best Stage 4 model).

import numpy as np
import pandas as pd
import glob
import pickle
from sklearn.decomposition import PCA

BASE = "/scratch/ariana.l/Stage 4 Link Prediction Model/"
STAGE5_BASE = "/scratch/ariana.l/Stage 5 PPE Representation Study/"
PPE_DIR = "/scratch/ariana.l/ppe-outputs/opportunity_surface/"

# --- Load common bins ---
print("Loading existence matrices...")
F = pd.read_csv(BASE + "stage4_F_existence_phenofield.csv", index_col=0)
P = pd.read_csv(BASE + "stage4_P_existence_gbif_combined.csv", index_col=0)

common_bins = sorted(set(F.columns) & set(P.columns))
print(f"Common bins: {len(common_bins)}")

# --- Load species_data from Stage 5.1 if still in memory, else rebuild ---
# Check if species_data exists; if not, reload V_delta CSV as a proxy
# We need species_data to compute mean norm per bin
try:
    _ = species_data
    print(f"species_data already in memory: {len(species_data)} species")
except NameError:
    print("species_data not in memory — will rebuild PMf from parquet files in Cell 2")

Loading existence matrices...
Common bins: 3160
species_data not in memory — will rebuild PMf from parquet files in Cell 2


In [2]:
# Stage 5 — PPE Representation Study
# 5.2 — Probability Flowering Matrix (B)
#
# Goal: Replace binary plant existence matrix F with PMf — a continuous
# spatiotemporal probability matrix where PMf[species, bin, week] = norm
# from PPE opportunity surface. PCA PMf to 15D → Vf_prob.
# Feature vector: [Vf_prob (15D), Vp (15D), N (1D)] = 31D
# Compare against A2 (binary Vf) and A3 (best Stage 4 model).

import numpy as np
import pandas as pd
import glob
import pickle
from collections import defaultdict
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/scratch/ariana.l/Stage 4 Link Prediction Model/"
STAGE5_BASE = "/scratch/ariana.l/Stage 5 PPE Representation Study/"
PPE_DIR = "/scratch/ariana.l/ppe-outputs/opportunity_surface/"

In [3]:
# Cell 2 — Load existence matrices and reconstruct common bins

print("Loading existence matrices...")
F = pd.read_csv(BASE + "stage4_F_existence_phenofield.csv", index_col=0)
P = pd.read_csv(BASE + "stage4_P_existence_gbif_combined.csv", index_col=0)

common_bins = sorted(set(F.columns) & set(P.columns))
bin_to_idx = {b: i for i, b in enumerate(common_bins)}
n_bins = len(common_bins)  # 3160
n_weeks = 52
print(f"Common bins: {n_bins}")

def snap(x):
    return round(round(x * 2) / 2, 1)

def fmt(x):
    return f"{x:.1f}"

Loading existence matrices...
Common bins: 3160


In [4]:
# Cell 3 — Read PPE opportunity surface and collect species data
# Identical to 5.1 — stores (bin_idx, week_idx, norm) per species

species_data = defaultdict(list)

files = sorted(glob.glob(PPE_DIR + "part_*.parquet"))
print(f"Found {len(files)} parquet files")
print("Reading opportunity surface files...")

for i, fpath in enumerate(files):
    df = pd.read_parquet(fpath, columns=['species', 'centroid_lat', 'centroid_lon', 'week', 'norm'])
    df['bin_key'] = df['centroid_lat'].map(snap).map(fmt) + '_' + df['centroid_lon'].map(snap).map(fmt)
    df = df[df['bin_key'].isin(bin_to_idx)]
    for row in df.itertuples(index=False):
        species_data[row.species].append((bin_to_idx[row.bin_key], int(row.week) - 1, row.norm))
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(files)} files, {len(species_data)} species seen so far")

print(f"\nDone. Species with PPE data: {len(species_data)}")

Found 6697 parquet files
Reading opportunity surface files...
  500/6697 files, 500 species seen so far
  1000/6697 files, 1000 species seen so far
  1500/6697 files, 1500 species seen so far
  2000/6697 files, 2000 species seen so far
  2500/6697 files, 2500 species seen so far
  3000/6697 files, 3000 species seen so far
  3500/6697 files, 3500 species seen so far
  4000/6697 files, 4000 species seen so far
  4500/6697 files, 4500 species seen so far
  5000/6697 files, 5000 species seen so far
  5500/6697 files, 5500 species seen so far
  6000/6697 files, 6000 species seen so far
  6500/6697 files, 6500 species seen so far

Done. Species with PPE data: 6697


In [5]:
# Cell 4 — Assemble PMf (n_species × n_bins × n_weeks) and fit PCA to 15D

plant_species = sorted(species_data.keys())
n_species = len(plant_species)
print(f"Plant species: {n_species}")

print(f"Allocating PMf: ({n_species}, {n_bins}, {n_weeks})...")
PMf = np.zeros((n_species, n_bins, n_weeks), dtype=np.float32)

print("Filling PMf...")
for sp_i, sp in enumerate(plant_species):
    for (bin_idx, week_idx, norm) in species_data[sp]:
        PMf[sp_i, bin_idx, week_idx] = norm
    if (sp_i + 1) % 1000 == 0:
        print(f"  {sp_i+1}/{n_species} species filled")

print(f"PMf shape: {PMf.shape}")
print(f"Memory usage: {PMf.nbytes / 1e9:.2f} GB")

# Flatten to (n_species, n_bins * n_weeks) for PCA
PMf_flat = PMf.reshape(n_species, n_bins * n_weeks)
print(f"Flattened shape: {PMf_flat.shape}")

# Fit PCA to 15D — same dimensionality as original Vf
print("Fitting PCA (randomized, 15 components)...")
pca_pmf = PCA(n_components=15, svd_solver='randomized', random_state=42)
Vf_prob = pca_pmf.fit_transform(PMf_flat)
print(f"Vf_prob shape: {Vf_prob.shape}")
print(f"Variance explained per component: {pca_pmf.explained_variance_ratio_.round(3)}")
print(f"Total variance explained: {pca_pmf.explained_variance_ratio_.sum():.3f}")

Plant species: 6697
Allocating PMf: (6697, 3160, 52)...
Filling PMf...
  1000/6697 species filled
  2000/6697 species filled
  3000/6697 species filled
  4000/6697 species filled
  5000/6697 species filled
  6000/6697 species filled
PMf shape: (6697, 3160, 52)
Memory usage: 4.40 GB
Flattened shape: (6697, 164320)
Fitting PCA (randomized, 15 components)...
Vf_prob shape: (6697, 15)
Variance explained per component: [0.201 0.109 0.052 0.025 0.019 0.016 0.011 0.01  0.009 0.007 0.006 0.005
 0.005 0.005 0.005]
Total variance explained: 0.486


In [8]:
# Cell 6 — Reconstruct training pairs and assemble 31D feature vectors

print("Loading assets...")
Vp_df = pd.read_csv(BASE + "stage4_Vp_gbif.csv", index_col=0)
globi = pd.read_csv(BASE + "stage4_globi_conus_broad.csv")

F_common = F[common_bins]
P_common = P[common_bins]

# Positive pairs
pos_pairs = globi[['sourceTaxonName', 'targetTaxonName']].drop_duplicates()
pos_pairs.columns = ['pollinator', 'plant']
pos_pairs = pos_pairs[
    pos_pairs['plant'].isin(Vf_prob_df.index) &
    pos_pairs['plant'].isin(F.index) &
    pos_pairs['pollinator'].isin(Vp_df.index) &
    pos_pairs['pollinator'].isin(P.index)
]
pos_pairs['label'] = 1
print(f"Positive pairs: {len(pos_pairs)}")

# Negative pairs
pos_set = set(zip(pos_pairs['pollinator'], pos_pairs['plant']))
all_plants = list(set(Vf_prob_df.index) & set(F.index))
all_pols = list(set(Vp_df.index) & set(P.index))

np.random.seed(42)
neg_pairs = []
while len(neg_pairs) < len(pos_pairs) * 3:
    pol = np.random.choice(all_pols)
    plant = np.random.choice(all_plants)
    if (pol, plant) not in pos_set:
        neg_pairs.append((pol, plant, 0))

neg_pairs = pd.DataFrame(neg_pairs, columns=['pollinator', 'plant', 'label'])
print(f"Negative pairs: {len(neg_pairs)}")

pairs = pd.concat([pos_pairs, neg_pairs], ignore_index=True)

# Assemble 31D feature vectors
def build_features(row):
    vf = Vf_prob_df.loc[row.plant].values     # 15D — spatiotemporal plant embedding
    vp = Vp_df.loc[row.pollinator].values     # 15D — binary pollinator embedding
    N  = float(np.dot(F_common.loc[row.plant].values, P_common.loc[row.pollinator].values))
    return np.concatenate([vf, vp, [N]])      # 31D

print("Assembling feature matrix...")
X = np.vstack([build_features(row) for row in pairs.itertuples()])
y = pairs['label'].values
print(f"X shape: {X.shape}, positive rate: {y.mean():.3f}")

Loading assets...
Positive pairs: 3074
Negative pairs: 9222
Assembling feature matrix...
X shape: (12296, 31), positive rate: 0.250


In [9]:
# Cell 7 — Train/test split and logistic regression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

clf_b = LogisticRegression(max_iter=1000, random_state=42)
clf_b.fit(X_train, y_train)

y_prob = clf_b.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, y_prob)
pr = average_precision_score(y_test, y_prob)

print(f"\nB (31D, PMf) Results:")
print(f"  ROC-AUC: {roc:.3f}")
print(f"  PR-AUC:  {pr:.3f}")
print(f"\nFor reference:")
print(f"  A2 (31D, binary Vf):        ROC-AUC 0.931, PR-AUC 0.842")
print(f"  A' (35D, V_delta appended): ROC-AUC 0.937, PR-AUC 0.855")
print(f"  A3 (32D, scalar delta):     ROC-AUC 0.950, PR-AUC 0.868")

Train: (9836, 31), Test: (2460, 31)

B (31D, PMf) Results:
  ROC-AUC: 0.938
  PR-AUC:  0.856

For reference:
  A2 (31D, binary Vf):        ROC-AUC 0.931, PR-AUC 0.842
  A' (35D, V_delta appended): ROC-AUC 0.937, PR-AUC 0.855
  A3 (32D, scalar delta):     ROC-AUC 0.950, PR-AUC 0.868


In [10]:
# Cell 8 — Save model

with open(STAGE5_BASE + "stage5_B_logistic.pkl", "wb") as f:
    pickle.dump(clf_b, f)
print("Saved clf_b")

Saved clf_b
